# INTRODUCTION

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">The Google Play Store is one of the world's largest digital marketplaces, hosting millions of applications across diverse categories and serving billions of Android users. Understanding the characteristics of successful applications can provide valuable insights for developers and businesses seeking to improve user engagement and market performance. This dataset contains information on Google Play Store applications, including attributes such as category, rating, size, number of installs, reviews, price, and content rating. Through Exploratory Data Analysis (EDA), this project aims to uncover patterns, trends, and relationships among these features to better understand the factors that influence an app's popularity and user satisfaction.

CREDIT: L. Gupta, "Google Play Store Apps," Feb 2019. Available: https://www.kaggle.com/lava18/google-play-store-apps

# OBJECTIVES:

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
<li>To analyze the distribution of applications across different categories on the Google Play Store.
<li>To identify trends and patterns in app ratings, reviews, installs, and pricing.
<li>To examine the relationship between app characteristics and their popularity among users.
<li>To compare the performance of free and paid applications across various metrics.
<li>To uncover actionable insights that can help developers make informed decisions when designing and marketing Android applications.

<div>

# 1. Getting Ready With Dataset

In [63]:
import numpy as np 
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so



In [44]:

data = pl.read_csv(
    "/home/kartika/Datasets/googleplaystore.csv", 
    schema_overrides={
        "Reviews": pl.String,
        "Price": pl.String,
        "Installs": pl.String
    }
)
data = data.to_pandas()

data["Price"] = data["Price"].str.replace("$", "", regex=False).str.strip()
data["Price"] = pd.to_numeric(data["Price"], errors="coerce")


data["Reviews"] = pd.to_numeric(data["Reviews"], errors="coerce")
data.drop(index = [10472, 10474], inplace = True)

data.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19M,"10,000+",Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,"500,000+",Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7M,"5,000,000+",Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25M,"50,000,000+",Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8M,"100,000+",Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Issue Identified: A column-shift anomaly was detected at row index 10474 (Life Made WI-Fi Touchscreen Photo Frame) due to a missing Category field. This shifted non-numeric strings into numerical columns (e.g., '3.0M' in Reviews, 'Everyone' in Price), causing schema parse failures.

Action Taken:

Overrode initial import schemas to ingest numeric target columns as raw strings.

Stripped currency formatting ($) and applied soft numeric coercion (pd.to_numeric(..., errors='coerce')) to turn misaligned strings into NaN values without interrupting pipeline execution.

Dropped the corrupted row to ensure dataset integrity across numeric analyses.

In [45]:
data.tail()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10836,Sya9a Maroc - FR,FAMILY,4.5,38.0,53M,"5,000+",Free,0.0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4.0,3.6M,100+,Free,0.0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10838,Parkinson Exercices FR,MEDICAL,NaN,3.0,9.5M,"1,000+",Free,0.0,Everyone,Medical,"January 20, 2017",1.0,2.2 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114.0,Varies with device,"1,000+",Free,0.0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device
10840,iHoroscope - 2018 Daily Horoscope & Astrology,LIFESTYLE,4.5,398307.0,19M,"10,000,000+",Free,0.0,Everyone,Lifestyle,"July 25, 2018",Varies with device,Varies with device


In [46]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10839 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10839 non-null  object 
 1   Category        10839 non-null  object 
 2   Rating          9365 non-null   float64
 3   Reviews         10839 non-null  float64
 4   Size            10839 non-null  object 
 5   Installs        10839 non-null  object 
 6   Type            10839 non-null  object 
 7   Price           10839 non-null  float64
 8   Content Rating  10839 non-null  object 
 9   Genres          10839 non-null  object 
 10  Last Updated    10839 non-null  object 
 11  Current Ver     10838 non-null  object 
 12  Android Ver     10839 non-null  object 
dtypes: float64(3), object(10)
memory usage: 1.2+ MB


In [47]:
data.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 0
Price                0
Content Rating       0
Genres               0
Last Updated         0
Current Ver          1
Android Ver          0
dtype: int64

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">There are some columns with missing values, and some with inconsistent and non-unifrom values, which are handled in following ways. 

<div>

In [ ]:
data['Rating'] = data['Rating'].fillna(data['Rating'].median())

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Since, 13.6% of the rows are null, instead of just filling with 0, filled it with median value to preserve distribution of data.

<div>

In [31]:
null_value = data.loc[data['Reviews'].isnull()]
null_value

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,NaN,"1,000+",Free,0,NaN,None,"February 11, 2018",1.0.19,4.0 and up,None


In [32]:
data['Reviews'] = data['Reviews'].fillna(0)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Since, only single value is missing, filled it with 0.

<di>

In [ ]:
data.loc[data['Size'] == 'Varies with device']

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
37,Floor Plan Creator,ART_AND_DESIGN,4.1,36639.0,Varies with device,"5,000,000+",Free,0.0,Everyone,Art & Design,"July 14, 2018",Varies with device,2.3.3 and up
42,Textgram - write on photos,ART_AND_DESIGN,4.4,295221.0,Varies with device,"10,000,000+",Free,0.0,Everyone,Art & Design,"July 30, 2018",Varies with device,Varies with device
52,Used Cars and Trucks for Sale,AUTO_AND_VEHICLES,4.6,17057.0,Varies with device,"1,000,000+",Free,0.0,Everyone,Auto & Vehicles,"July 30, 2018",Varies with device,Varies with device
67,Ulysse Speedometer,AUTO_AND_VEHICLES,4.3,40211.0,Varies with device,"5,000,000+",Free,0.0,Everyone,Auto & Vehicles,"July 30, 2018",Varies with device,Varies with device
68,REPUVE,AUTO_AND_VEHICLES,3.9,356.0,Varies with device,"100,000+",Free,0.0,Everyone,Auto & Vehicles,"May 25, 2018",Varies with device,Varies with device
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10713,My Earthquake Alerts - US & Worldwide Earthquakes,WEATHER,4.4,3471.0,Varies with device,"100,000+",Free,0.0,Everyone,Weather,"July 24, 2018",Varies with device,Varies with device
10725,Posta App,MAPS_AND_NAVIGATION,3.6,8.0,Varies with device,"1,000+",Free,0.0,Everyone,Maps & Navigation,"September 27, 2017",Varies with device,4.4 and up
10765,Chat For Strangers - Video Chat,SOCIAL,3.4,622.0,Varies with device,"100,000+",Free,0.0,Mature 17+,Social,"May 23, 2018",Varies with device,Varies with device
10826,Frim: get new friends on local chat rooms,SOCIAL,4.0,88486.0,Varies with device,"5,000,000+",Free,0.0,Mature 17+,Social,"March 23, 2018",Varies with device,Varies with device


In [51]:
def parse_size(val):
    if pd.isna(val) or val == 'Varies with device':
        return np.nan
    
    val = str(val).strip().upper()
    
    if 'M' in val:
        return float(val.replace('M', ''))
    
    elif 'K' in val:
        return float(val.replace('K', '')) / 1024
   
    elif 'G' in val:
        return float(val.replace('G', '')) * 1024

    elif '1,000+' in val:
        return 1
    


data['Size'] = data['Size'].apply(parse_size)

In [52]:
data['Size'] = data.groupby('Category')['Size'].transform(lambda x: x.fillna(x.median()))

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Replaced 'Varies with device', non standard record with np.nan. One record with '1,000+' is replaced by 1 MB. 
<br>
Standardised unit multipliers (M for Megabytes, k for Kilobytes), and converted to a unified numeric unit like MB.
<br>
App sizes usually correlate strongly with their Category (e.g., Games are usually much larger than Tools). Imputed by category median.

<div>

In [56]:

data['Installs'] = (
    data['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)


data['Installs'] = pd.to_numeric(data['Installs'], errors='coerce')


data['Installs'] = data['Installs'].fillna(0).astype(int)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
Stripped non-numeric formatting symbols—specifically commas (,) and plus signs (+)—to retrieve raw numeric strings (e.g., '100000+ '10000+).
<br> Converted the column data type from object (string) to int64 to enable quantitative analysis and aggregation.

<div>

In [59]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10839 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10839 non-null  object 
 1   Category        10839 non-null  object 
 2   Rating          10839 non-null  float64
 3   Reviews         10839 non-null  float64
 4   Size            10839 non-null  float64
 5   Installs        10839 non-null  int64  
 6   Type            10839 non-null  object 
 7   Price           10839 non-null  float64
 8   Content Rating  10839 non-null  object 
 9   Genres          10839 non-null  object 
 10  Last Updated    10839 non-null  object 
 11  Current Ver     10838 non-null  object 
 12  Android Ver     10839 non-null  object 
dtypes: float64(4), int64(1), object(8)
memory usage: 1.2+ MB


In [60]:
data.drop(columns = ['Last Updated', 'Current Ver', 'Android Ver'], inplace = True)

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">Dropped least important fields.

<div>

## Univariate Analysis